# dyn_A_g05 EOS 抑制 A/B 对比可视化 (2 配置 × 50 samples)

| 标签 | 模式 | eos_penalty | 说明 |
|------|------|-------------|------|
| dyn_A_g05 | dynamic, gamma=0.5 | 0 (关) | baseline，无 EOS 抑制 |
| dyn_A_g05_EOF | dynamic, gamma=0.5 | 10.0 | 开 EOS 抑制 |

## 1. 环境设置

In [ ]:
import os, sys, gc
import torch
import torch.nn.functional as F
import numpy as np

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['HF_ALLOW_CODE_EVAL'] = '1'
os.environ['HF_DATASETS_TRUST_REMOTE_CODE'] = 'true'

NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
os.chdir(os.path.join(NOTEBOOK_DIR, 'llada'))
print(f'Working dir: {os.getcwd()}')

if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

torch.cuda.empty_cache(); gc.collect()
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

## 2. 模型 & 数据加载

In [ ]:
from transformers import AutoTokenizer, AutoConfig
from model.modeling_llada import LLaDAModelLM
from datasets import load_dataset

MODEL_PATH = 'GSAI-ML/LLaDA-8B-Instruct'
MASK_ID = 126336

config = AutoConfig.from_pretrained(MODEL_PATH)
config.flash_attention = True
model = LLaDAModelLM.from_pretrained(
    MODEL_PATH, trust_remote_code=True, torch_dtype=torch.bfloat16, config=config,
).eval().to('cuda')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

gsm8k = load_dataset('gsm8k', 'main', split='test')
print(f'Model loaded. GSM8K test: {len(gsm8k)} samples')

In [ ]:
FEW_SHOT_EXAMPLES = """Question: Jen and Tyler are gymnasts practicing flips. Jen is practicing the triple-flip while Tyler is practicing the double-flip. Jen did sixteen triple-flips during practice. Tyler flipped in the air half the number of times Jen did. How many double-flips did Tyler do?
Answer: Jen did 16 triple-flips, so she did 16 * 3 = <<16*3=48>>48 flips.
Tyler did half the number of flips, so he did 48 / 2 = <<48/2=24>>24 flips.
A double flip has two flips, so Tyler did 24 / 2 = <<24/2=12>>12 double-flips.
#### 12

Question: Four people in a law firm are planning a party. Mary will buy a platter of pasta for $20 and a loaf of bread for $2. Elle and Andrea will split the cost for buying 4 cans of soda which cost $1.50 each, and chicken wings for $10. Joe will buy a cake that costs $5. How much more will Mary spend than the rest of the firm put together?
Answer: Mary will spend $20 + $2 = $<<20+2=22>>22.
Elle and Andrea will spend $1.5 x 4 = $<<1.5*4=6>>6 for the soda.
Elle and Andrea will spend $6 + $10 = $<<6+10=16>>16 for the soda and chicken wings.
Elle, Andrea, and Joe together will spend $16 + $5 = $<<16+5=21>>21.
So, Mary will spend $22 - $21 = $<<22-21=1>>1 more than all of them combined.
#### 1

Question: A charcoal grill burns fifteen coals to ash every twenty minutes of grilling. The grill ran for long enough to burn three bags of coals. Each bag of coal contains 60 coals. How long did the grill run?
Answer: The grill burned 3 * 60 = <<3*60=180>>180 coals.
It takes 20 minutes to burn 15 coals, so the grill ran for 180 / 15 * 20 = <<180/15*20=240>>240 minutes.
#### 240

Question: A bear is preparing to hibernate for the winter and needs to gain 1000 pounds. At the end of summer, the bear feasts on berries and small woodland animals. During autumn, it devours acorns and salmon. It gained a fifth of the weight it needed from berries during summer, and during autumn, it gained twice that amount from acorns. Salmon made up half of the remaining weight it had needed to gain. How many pounds did it gain eating small animals?
Answer: The bear gained 1 / 5 * 1000 = <<1/5*1000=200>>200 pounds from berries.
It gained 2 * 200 = <<2*200=400>>400 pounds from acorns.
It still needed 1000 - 200 - 400 = <<1000-200-400=400>>400 pounds.
Thus, it gained 400 / 2 = <<400/2=200>>200 pounds from salmon.
Therefore, the bear gained 400 - 200 = <<400-200=200>>200 pounds from small animals.
#### 200

Question: Brendan can cut 8 yards of grass per day, he bought a lawnmower and it helped him to cut more yards by Fifty percent per day. How many yards will Brendan be able to cut after a week?
Answer: The additional yard Brendan can cut after buying the lawnmower is 8 x 0.50 = <<8*0.50=4>>4 yards.
So, the total yards he can cut with the lawnmower is 8 + 4 = <<8+4=12>>12.
Therefore, the total number of yards he can cut in a week is 12 x 7 = <<12*7=84>>84 yards.
#### 84"""


def build_prompt(question: str) -> torch.Tensor:
    text = FEW_SHOT_EXAMPLES + f'\n\nQuestion: {question}\nAnswer:'
    messages = [{'role': 'user', 'content': text}]
    formatted = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    ids = tokenizer(formatted)['input_ids']
    return torch.tensor(ids, dtype=torch.long, device='cuda').unsqueeze(0)


import re

def extract_answer(text: str) -> str | None:
    m = re.search(r'####\s*(-?[\d,]+\.?\d*)', text)
    return m.group(1).replace(',', '').strip() if m else None


LIMIT = 50
prompts = [build_prompt(gsm8k[i]['question']) for i in range(LIMIT)]
questions = [gsm8k[i]['question'] for i in range(LIMIT)]
ref_answers = [gsm8k[i]['answer'] for i in range(LIMIT)]
print(f'Built {len(prompts)} prompts, first length: {prompts[0].shape[1]} tokens')

## 3. 核心函数

直接复用 `viz_saber_denoising.ipynb` 中的 `generate_with_collection_saber` 和 `process_sample_saber`。
唯一区别：`generate_with_collection_saber` 新增 `fullstage_berm` 和 `post_global_berm_rounds` 参数。

In [ ]:
import math as _math
from generate import add_gumbel_noise
from viz_static import process_sample as _base_process_sample, generate_multi_html


def _apply_eos_bias(logits, m, eos_penalty, eos_token_id):
    """Suppress EOS logit proportionally to remaining mask ratio m.

    m=1.0 (all masked) → heavy penalty; m=0.0 (done) → no penalty.
    """
    if eos_penalty == 0.0 or eos_token_id is None:
        return logits
    _eps = 1e-3
    bias = eos_penalty * _math.log(max(1.0 - m + _eps, _eps))
    logits = logits.clone()
    logits[..., eos_token_id] += bias
    return logits


def _saber_select_viz(confidence, mask, conf_sum, conf_count, saber_n):
    n_masked = int(mask.sum().item())
    if n_masked == 0:
        empty = torch.empty(0, dtype=torch.long, device=confidence.device)
        return empty, 0.0, empty

    if conf_count > 0:
        tau = (conf_sum / conf_count).item()
    else:
        tau = confidence[mask].max().item()

    cand = torch.where(confidence >= tau)[0]
    sel = cand
    forced_sel = torch.empty(0, dtype=torch.long, device=confidence.device)
    if sel.numel() < saber_n:
        k = min(saber_n, n_masked)
        _, sel = torch.topk(confidence, k=k)
        if cand.numel() > 0:
            forced_sel = sel[~torch.isin(sel, cand)]
        else:
            forced_sel = sel

    return sel, tau, forced_sel


def _berm_cross_step_viz(x_block, full_conf, last_conf, unmask_time_conf,
                          conf_sum, conf_count, n_unmask, saber_n, saber_mu, mask_id):
    still_masked = (x_block == mask_id)
    delta = full_conf - last_conf
    delta_rem = delta.clone()
    delta_rem[still_masked] = float('inf')

    n_eligible = int((~still_masked).sum().item())
    mu_t = max(saber_n // 2, (n_unmask + saber_mu - 1) // saber_mu)
    mu_t = min(mu_t, max(0, n_unmask - 1))
    mu_t = min(mu_t, n_eligible)
    if mu_t <= 0:
        return 0, conf_sum, conf_count, []

    _, rem_idx = torch.topk(delta_rem, k=mu_t, largest=False)
    remasked = 0
    rem_idx_local = []
    for ri in rem_idx:
        ri_v = ri.item()
        if x_block[ri_v] != mask_id:
            x_block[ri_v] = mask_id
            conf_sum -= unmask_time_conf[ri_v]
            conf_count -= 1
            unmask_time_conf[ri_v] = 0.0
            remasked += 1
            rem_idx_local.append(ri_v)
    return remasked, conf_sum, conf_count, rem_idx_local


@torch.no_grad()
def generate_with_collection_saber(
    model, prompt, steps=256, gen_length=256, block_length=32,
    temperature=0., mask_id=126336,
    mid_trigger_ratio=0.8, saber_n=4, saber_mu=8, global_aadu=True,
    fullstage_berm=False, post_global_berm_rounds=0,
    berm_scope='window',
    eos_penalty=0.0, eos_token_id=None,
    verbose=False, sample_idx=None,
):
    B = prompt.shape[0]
    Lp = int(prompt.shape[1])
    seq_len = Lp + gen_length
    device = model.device
    assert gen_length % block_length == 0
    num_blocks = gen_length // block_length
    trigger_thresh = int(block_length * mid_trigger_ratio)
    NEG_INF = torch.tensor(float('-inf'), device=device, dtype=torch.float64)

    x = torch.full((B, seq_len), mask_id, dtype=torch.long, device=device)
    x[:, :Lp] = prompt
    nfe = 0

    g_conf_sum = torch.zeros(B, device=device, dtype=torch.float64)
    g_conf_count = torch.zeros(B, device=device, dtype=torch.long)
    pconf = torch.zeros(B, seq_len, dtype=torch.float64, device=device)

    history = []
    global_step = 0

    def _berm_left(s, e, watching_nb):
        if berm_scope == 'window':
            return 0
        elif berm_scope == 'current_block':
            return Lp + watching_nb * block_length - s
        elif berm_scope == 'trail_block':
            return Lp + max(0, watching_nb - 1) * block_length - s
        return 0

    def _scoped_berm_viz(x_slice_j, fc, lc, utc, cs, cc, nu, left_off):
        if left_off <= 0:
            return _berm_cross_step_viz(x_slice_j, fc, lc, utc, cs, cc, nu, saber_n, saber_mu, mask_id)
        saved_utc = utc[:left_off].clone()
        utc[:left_off] = 0.0
        saved_x = x_slice_j[:left_off].clone()
        result = _berm_cross_step_viz(x_slice_j, fc, lc, utc, cs, cc, nu, saber_n, saber_mu, mask_id)
        x_slice_j[:left_off] = saved_x
        utc[:left_off] = saved_utc
        return result

    def _snap(step_i, nb, s, e, x0_full_seq, forced_floor_pos=None, berm_remask_pos=None):
        mask_now = (x == mask_id)
        return {
            'step': step_i, 'block_id': nb,
            'x': x.cpu().numpy().copy(),
            'x0': x0_full_seq.cpu().numpy().copy(),
            'mask': mask_now.cpu().numpy().copy(),
            'confidence': pconf.cpu().numpy().copy(),
            'current_block_range': (s, e),
            'forced_floor_pos': forced_floor_pos or [[] for _ in range(B)],
            'berm_remask_pos': berm_remask_pos or [[] for _ in range(B)],
        }

    nb = 0
    while nb < num_blocks:
        s = Lp + nb * block_length
        e = s + block_length
        block_steps = []

        if global_aadu:
            conf_sum = g_conf_sum.clone()
            conf_count = g_conf_count.clone()
        else:
            conf_sum = torch.zeros(B, device=device, dtype=torch.float64)
            conf_count = torch.zeros(B, device=device, dtype=torch.long)

        BL = e - s
        last_conf = torch.zeros(B, BL, device=device, dtype=torch.float64)
        unmask_tc = torch.zeros(B, BL, device=device, dtype=torch.float64)

        # Phase 1: warm-up
        out = model(x, use_cache=True)
        past_kv = out.past_key_values
        nfe += 1
        _m_warm = float((x[:, s:e] == mask_id).sum().item()) / (B * (e - s))
        blk_logits = _apply_eos_bias(out.logits[:, s:e, :], _m_warm, eos_penalty, eos_token_id)
        del out

        x0 = torch.argmax(add_gumbel_noise(blk_logits, temperature), dim=-1)
        p = F.softmax(blk_logits.to(torch.float64), dim=-1)
        x0_p = torch.gather(p, -1, x0.unsqueeze(-1)).squeeze(-1)
        blk_mask = (x[:, s:e] == mask_id)

        forced_floor_pos = [[] for _ in range(B)]
        for j in range(B):
            conf_j = torch.where(blk_mask[j], x0_p[j], NEG_INF)
            sel, _, forced_sel = _saber_select_viz(conf_j, blk_mask[j], conf_sum[j], conf_count[j], saber_n)
            if sel.numel() == 0:
                continue
            x[j, s + sel] = x0[j, sel]
            sc = x0_p[j, sel]
            conf_sum[j] += sc.sum()
            conf_count[j] += sel.numel()
            unmask_tc[j, sel] = sc
            if forced_sel.numel() > 0:
                forced_floor_pos[j].extend([int(s + k.item()) for k in forced_sel])

        if fullstage_berm:
            berm_remask_pos_warmup = [[] for _ in range(B)]
            for j in range(B):
                n_unm = int((x[j, s:e] != mask_id).sum().item())
                if n_unm <= 1:
                    continue
                _, conf_sum[j], conf_count[j], rem_local = _berm_cross_step_viz(
                    x[j, s:e], x0_p[j], torch.zeros_like(x0_p[j]), unmask_tc[j],
                    conf_sum[j], conf_count[j], max(saber_n, n_unm), saber_n, saber_mu, mask_id,
                )
                if rem_local:
                    berm_remask_pos_warmup[j].extend([int(s + k) for k in rem_local])

        last_conf = x0_p.clone()
        pconf[:, s:e] = x0_p

        x0_snap = x.clone()
        x0_snap[:, s:e] = x0
        block_steps.append(_snap(global_step, nb, s, e, x0_snap, forced_floor_pos=forced_floor_pos))
        global_step += 1

        # Phase 2: refinement with expand
        rp = torch.zeros(B, seq_len, dtype=torch.bool, device=device)
        rp[:, s:e] = True
        watching_nb = nb
        blocks_consumed = 1

        for _step in range(steps):
            if not (x[:, s:e] == mask_id).any():
                break

            can_expand = (watching_nb + 1 < num_blocks)
            if can_expand:
                wb_s = Lp + watching_nb * block_length
                wb_e = wb_s + block_length
                remaining = int((x[:, wb_s:wb_e] == mask_id).sum(dim=1).max().item())
                if remaining <= trigger_thresh:
                    next_nb = watching_nb + 1
                    e_new = min(Lp + (next_nb + 1) * block_length, seq_len)
                    e = e_new
                    BL = e - s

                    if not global_aadu:
                        conf_sum = torch.zeros(B, device=device, dtype=torch.float64)
                        conf_count = torch.zeros(B, device=device, dtype=torch.long)

                    out = model(x, use_cache=True)
                    past_kv = out.past_key_values
                    nfe += 1
                    _m_exp = float((x[:, s:e] == mask_id).sum().item()) / (B * (e - s))
                    blk_logits = _apply_eos_bias(out.logits[:, s:e, :], _m_exp, eos_penalty, eos_token_id)
                    del out

                    x0 = torch.argmax(add_gumbel_noise(blk_logits, temperature), dim=-1)
                    p = F.softmax(blk_logits.to(torch.float64), dim=-1)
                    x0_p = torch.gather(p, -1, x0.unsqueeze(-1)).squeeze(-1)
                    blk_mask = (x[:, s:e] == mask_id)

                    forced_floor_pos = [[] for _ in range(B)]
                    for j in range(B):
                        conf_j = torch.where(blk_mask[j], x0_p[j], NEG_INF)
                        sel, _, forced_sel = _saber_select_viz(conf_j, blk_mask[j], conf_sum[j], conf_count[j], saber_n)
                        if sel.numel() == 0:
                            continue
                        x[j, s + sel] = x0[j, sel]
                        sc = x0_p[j, sel]
                        conf_sum[j] += sc.sum()
                        conf_count[j] += sel.numel()
                        if forced_sel.numel() > 0:
                            forced_floor_pos[j].extend([int(s + k.item()) for k in forced_sel])

                    last_conf = torch.zeros(B, BL, device=device, dtype=torch.float64)
                    unmask_tc = torch.zeros(B, BL, device=device, dtype=torch.float64)
                    last_conf[:, :x0_p.shape[1]] = x0_p
                    for j in range(B):
                        unmasked_j = (x[j, s:e] != mask_id)
                        unmask_tc[j, :unmasked_j.shape[0]][unmasked_j] = x0_p[j][unmasked_j]

                    rp = torch.zeros(B, seq_len, dtype=torch.bool, device=device)
                    rp[:, s:e] = True
                    blocks_consumed += 1
                    watching_nb = next_nb

                    pconf[:, s:e] = x0_p
                    x0_snap = x.clone()
                    x0_snap[:, s:e] = x0
                    block_steps.append(_snap(global_step, nb, s, e, x0_snap, forced_floor_pos=forced_floor_pos))
                    global_step += 1
                    continue

            _m_ref = float((x[:, s:e] == mask_id).sum().item()) / (B * (e - s))
            blk_logits = _apply_eos_bias(model(
                x[:, s:e], past_key_values=past_kv,
                use_cache=True, replace_position=rp,
            ).logits, _m_ref, eos_penalty, eos_token_id)
            nfe += 1

            x0 = torch.argmax(add_gumbel_noise(blk_logits, temperature), dim=-1)
            p = F.softmax(blk_logits.to(torch.float64), dim=-1)
            x0_p = torch.gather(p, -1, x0.unsqueeze(-1)).squeeze(-1)
            blk_mask = (x[:, s:e] == mask_id)
            full_conf = x0_p.clone()

            forced_floor_pos = [[] for _ in range(B)]
            berm_remask_pos = [[] for _ in range(B)]
            for j in range(B):
                conf_j = torch.where(blk_mask[j], x0_p[j], NEG_INF)
                sel, _, forced_sel = _saber_select_viz(conf_j, blk_mask[j], conf_sum[j], conf_count[j], saber_n)
                n_unmask = sel.numel()
                if n_unmask == 0:
                    continue
                x[j, s + sel] = x0[j, sel]
                sc = x0_p[j, sel]
                conf_sum[j] += sc.sum()
                conf_count[j] += sel.numel()
                unmask_tc[j, sel] = sc

                if forced_sel.numel() > 0:
                    forced_floor_pos[j].extend([int(s + k.item()) for k in forced_sel])

                left_off = max(0, _berm_left(s, e, watching_nb))
                _, conf_sum[j], conf_count[j], rem_idx_local = _scoped_berm_viz(
                    x[j, s:e], full_conf[j], last_conf[j], unmask_tc[j],
                    conf_sum[j], conf_count[j], n_unmask, left_off,
                )
                if rem_idx_local:
                    berm_remask_pos[j].extend([int(s + k) for k in rem_idx_local])

            last_conf = full_conf.clone()
            pconf[:, s:e] = torch.where(blk_mask, x0_p, pconf[:, s:e])

            x0_snap = x.clone()
            x0_snap[:, s:e] = torch.where(blk_mask, x0, x[:, s:e])
            block_steps.append(_snap(
                global_step, nb, s, e, x0_snap,
                forced_floor_pos=forced_floor_pos,
                berm_remask_pos=berm_remask_pos,
            ))
            global_step += 1

        if global_aadu:
            g_conf_sum = conf_sum.clone()
            g_conf_count = conf_count.clone()

        history.append({'block_id': nb, 'steps': block_steps})
        nb += blocks_consumed

    return x, nfe, history


def process_sample_saber(sample_blocks, tokenizer, gen_length, block_length,
                          batch_idx=0, question=None, gen_answer=None, ref_answer=None):
    result = _base_process_sample(
        sample_blocks, tokenizer, batch_idx=batch_idx,
        question=question, gen_answer=gen_answer, ref_answer=ref_answer,
    )
    gen_start = int(result['gen_start'])
    num_blocks = gen_length // block_length
    result['num_blocks'] = num_blocks
    result['block_ranges'] = [
        [gen_start + i * block_length, gen_start + (i + 1) * block_length]
        for i in range(num_blocks)
    ]
    step_idx = 0
    for block in sample_blocks:
        for sd in block['steps']:
            ff_raw = sd.get('forced_floor_pos')
            br_raw = sd.get('berm_remask_pos')
            ff_set = set(int(p) for p in ff_raw[batch_idx]) if isinstance(ff_raw, list) and len(ff_raw) > batch_idx and isinstance(ff_raw[batch_idx], list) else set()
            br_set = set(int(p) for p in br_raw[batch_idx]) if isinstance(br_raw, list) and len(br_raw) > batch_idx and isinstance(br_raw[batch_idx], list) else set()
            for tk in result['steps'][step_idx]['tk']:
                pos = int(tk.get('p', -1))
                tk['f4'] = (pos in ff_set)
                tk['brm'] = (pos in br_set)
            step_idx += 1
    return result


@torch.no_grad()
def generate_with_collection_dynamic(
    model, prompt, steps=256, gen_length=256, block_length=32,
    temperature=0., mask_id=126336,
    mid_trigger_ratio=0.8, global_aadu=True,
    n_hi=8, n_lo=3, mu_lo=4, mu_hi=16,
    gamma_n=1.0, gamma_mu=1.0, gamma_floor=1.5,
    remask_ratio_hi=0.35, floor_lo=0,
    berm_scope='trail_block',
    eos_penalty=0.0, eos_token_id=None,
    verbose=False, sample_idx=None,
):
    import math
    B = prompt.shape[0]
    Lp = int(prompt.shape[1])
    seq_len = Lp + gen_length
    device = model.device
    assert gen_length % block_length == 0
    num_blocks = gen_length // block_length
    trigger_thresh = int(block_length * mid_trigger_ratio)
    NEG_INF = torch.tensor(float('-inf'), device=device, dtype=torch.float64)

    x = torch.full((B, seq_len), mask_id, dtype=torch.long, device=device)
    x[:, :Lp] = prompt
    nfe = 0
    g_conf_sum = torch.zeros(B, device=device, dtype=torch.float64)
    g_conf_count = torch.zeros(B, device=device, dtype=torch.long)
    pconf = torch.zeros(B, seq_len, dtype=torch.float64, device=device)
    history = []
    global_step = 0

    def _m(s, e):
        wl = e - s
        return float((x[:, s:e] == mask_id).sum().item()) / (B * wl) if wl > 0 else 0.0

    def _dn(m_): return max(1, round(n_lo + (n_hi - n_lo) * (m_ ** gamma_n)))
    def _dmu(m_): return max(1, round(mu_lo + (mu_hi - mu_lo) * ((1 - m_) ** gamma_mu)))
    def _dfl(m_, cn): return max(floor_lo, round(remask_ratio_hi * cn * (m_ ** gamma_floor)))

    def _dyn_berm(x_blk, fc, lc, utc, cs, cc, nu, cur_mu, cur_fl):
        still = (x_blk == mask_id)
        delta = fc - lc
        dr = delta.clone()
        dr[still] = float('inf')
        ne = int((~still).sum().item())
        mt = max(cur_fl, math.ceil(nu / cur_mu))
        mt = min(mt, max(0, nu - 1), ne)
        if mt <= 0:
            return 0, cs, cc, []
        _, ri = torch.topk(dr, k=mt, largest=False)
        rm = 0
        rl = []
        for r in ri:
            rv = r.item()
            if x_blk[rv] != mask_id:
                x_blk[rv] = mask_id
                cs -= utc[rv]; cc -= 1; utc[rv] = 0.0
                rm += 1; rl.append(rv)
        return rm, cs, cc, rl

    def _bl(s, e, wnb):
        if berm_scope == 'trail_block':
            return Lp + max(0, wnb - 1) * block_length - s
        elif berm_scope == 'current_block':
            return Lp + wnb * block_length - s
        return 0

    def _snap(si, nb, s, e, x0s, ffp=None, brp=None):
        return {
            'step': si, 'block_id': nb,
            'x': x.cpu().numpy().copy(),
            'x0': x0s.cpu().numpy().copy(),
            'mask': (x == mask_id).cpu().numpy().copy(),
            'confidence': pconf.cpu().numpy().copy(),
            'current_block_range': (s, e),
            'forced_floor_pos': ffp or [[] for _ in range(B)],
            'berm_remask_pos': brp or [[] for _ in range(B)],
        }

    nb = 0
    while nb < num_blocks:
        s = Lp + nb * block_length
        e = s + block_length
        block_steps = []
        if global_aadu:
            conf_sum = g_conf_sum.clone(); conf_count = g_conf_count.clone()
        else:
            conf_sum = torch.zeros(B, device=device, dtype=torch.float64)
            conf_count = torch.zeros(B, device=device, dtype=torch.long)
        BL = e - s
        last_conf = torch.zeros(B, BL, device=device, dtype=torch.float64)
        unmask_tc = torch.zeros(B, BL, device=device, dtype=torch.float64)

        m_now = _m(s, e)
        cur_n = _dn(m_now)
        out = model(x, use_cache=True)
        past_kv = out.past_key_values; nfe += 1
        blk_logits = _apply_eos_bias(out.logits[:, s:e, :], m_now, eos_penalty, eos_token_id); del out
        x0 = torch.argmax(add_gumbel_noise(blk_logits, temperature), dim=-1)
        p = F.softmax(blk_logits.to(torch.float64), dim=-1)
        x0_p = torch.gather(p, -1, x0.unsqueeze(-1)).squeeze(-1)
        blk_mask = (x[:, s:e] == mask_id)
        ffp = [[] for _ in range(B)]
        for j in range(B):
            cj = torch.where(blk_mask[j], x0_p[j], NEG_INF)
            sel, _, fsel = _saber_select_viz(cj, blk_mask[j], conf_sum[j], conf_count[j], cur_n)
            if sel.numel() == 0: continue
            x[j, s + sel] = x0[j, sel]
            sc = x0_p[j, sel]; conf_sum[j] += sc.sum(); conf_count[j] += sel.numel()
            unmask_tc[j, sel] = sc
            if fsel.numel() > 0: ffp[j].extend([int(s + k.item()) for k in fsel])
        last_conf = x0_p.clone(); pconf[:, s:e] = x0_p
        x0s = x.clone(); x0s[:, s:e] = x0
        block_steps.append(_snap(global_step, nb, s, e, x0s, ffp=ffp)); global_step += 1

        rp = torch.zeros(B, seq_len, dtype=torch.bool, device=device); rp[:, s:e] = True
        watching_nb = nb; blocks_consumed = 1

        for _step in range(steps):
            if not (x[:, s:e] == mask_id).any(): break
            m_now = _m(s, e); cur_n = _dn(m_now); cur_mu = _dmu(m_now); cur_fl = _dfl(m_now, cur_n)
            can_expand = (watching_nb + 1 < num_blocks)
            if can_expand:
                wb_s = Lp + watching_nb * block_length; wb_e = wb_s + block_length
                rem = int((x[:, wb_s:wb_e] == mask_id).sum(dim=1).max().item())
                if rem <= trigger_thresh:
                    next_nb = watching_nb + 1
                    e = min(Lp + (next_nb + 1) * block_length, seq_len); BL = e - s
                    if not global_aadu:
                        conf_sum = torch.zeros(B, device=device, dtype=torch.float64)
                        conf_count = torch.zeros(B, device=device, dtype=torch.long)
                    out = model(x, use_cache=True); past_kv = out.past_key_values; nfe += 1
                    m_now = _m(s, e)
                    blk_logits = _apply_eos_bias(out.logits[:, s:e, :], m_now, eos_penalty, eos_token_id); del out
                    cur_n = _dn(m_now)
                    x0 = torch.argmax(add_gumbel_noise(blk_logits, temperature), dim=-1)
                    p = F.softmax(blk_logits.to(torch.float64), dim=-1)
                    x0_p = torch.gather(p, -1, x0.unsqueeze(-1)).squeeze(-1)
                    blk_mask = (x[:, s:e] == mask_id)
                    ffp = [[] for _ in range(B)]
                    for j in range(B):
                        cj = torch.where(blk_mask[j], x0_p[j], NEG_INF)
                        sel, _, fsel = _saber_select_viz(cj, blk_mask[j], conf_sum[j], conf_count[j], cur_n)
                        if sel.numel() == 0: continue
                        x[j, s + sel] = x0[j, sel]
                        sc = x0_p[j, sel]; conf_sum[j] += sc.sum(); conf_count[j] += sel.numel()
                        if fsel.numel() > 0: ffp[j].extend([int(s + k.item()) for k in fsel])
                    last_conf = torch.zeros(B, BL, device=device, dtype=torch.float64)
                    unmask_tc = torch.zeros(B, BL, device=device, dtype=torch.float64)
                    last_conf[:, :x0_p.shape[1]] = x0_p
                    for j in range(B):
                        um = (x[j, s:e] != mask_id); unmask_tc[j, :um.shape[0]][um] = x0_p[j][um]
                    rp = torch.zeros(B, seq_len, dtype=torch.bool, device=device); rp[:, s:e] = True
                    blocks_consumed += 1; watching_nb = next_nb
                    pconf[:, s:e] = x0_p; x0s = x.clone(); x0s[:, s:e] = x0
                    block_steps.append(_snap(global_step, nb, s, e, x0s, ffp=ffp)); global_step += 1
                    continue

            blk_logits = _apply_eos_bias(
                model(x[:, s:e], past_key_values=past_kv, use_cache=True, replace_position=rp).logits,
                m_now, eos_penalty, eos_token_id)
            nfe += 1
            x0 = torch.argmax(add_gumbel_noise(blk_logits, temperature), dim=-1)
            p = F.softmax(blk_logits.to(torch.float64), dim=-1)
            x0_p = torch.gather(p, -1, x0.unsqueeze(-1)).squeeze(-1)
            blk_mask = (x[:, s:e] == mask_id); full_conf = x0_p.clone()
            ffp = [[] for _ in range(B)]; brp = [[] for _ in range(B)]
            lo = max(0, _bl(s, e, watching_nb))
            for j in range(B):
                cj = torch.where(blk_mask[j], x0_p[j], NEG_INF)
                sel, _, fsel = _saber_select_viz(cj, blk_mask[j], conf_sum[j], conf_count[j], cur_n)
                nu = sel.numel()
                if nu == 0: continue
                x[j, s + sel] = x0[j, sel]
                sc = x0_p[j, sel]; conf_sum[j] += sc.sum(); conf_count[j] += sel.numel()
                unmask_tc[j, sel] = sc
                if fsel.numel() > 0: ffp[j].extend([int(s + k.item()) for k in fsel])
                if lo <= 0:
                    _, conf_sum[j], conf_count[j], rl = _dyn_berm(
                        x[j, s:e], full_conf[j], last_conf[j], unmask_tc[j],
                        conf_sum[j], conf_count[j], nu, cur_mu, cur_fl)
                else:
                    su = unmask_tc[j][:lo].clone(); unmask_tc[j][:lo] = 0.0; sx = x[j, s:s+lo].clone()
                    _, conf_sum[j], conf_count[j], rl = _dyn_berm(
                        x[j, s:e], full_conf[j], last_conf[j], unmask_tc[j],
                        conf_sum[j], conf_count[j], nu, cur_mu, cur_fl)
                    x[j, s:s+lo] = sx; unmask_tc[j][:lo] = su
                if rl: brp[j].extend([int(s + k) for k in rl])
            last_conf = full_conf.clone()
            pconf[:, s:e] = torch.where(blk_mask, x0_p, pconf[:, s:e])
            x0s = x.clone(); x0s[:, s:e] = torch.where(blk_mask, x0, x[:, s:e])
            block_steps.append(_snap(global_step, nb, s, e, x0s, ffp=ffp, brp=brp)); global_step += 1

        if global_aadu: g_conf_sum = conf_sum.clone(); g_conf_count = conf_count.clone()
        history.append({'block_id': nb, 'steps': block_steps}); nb += blocks_consumed

    return x, nfe, history


print('Functions defined.')

## 4. 配置定义

In [ ]:
EOS_PENALTY = 10.0
EOS_TOKEN_ID = 151643   # Qwen tokenizer <|endoftext|> — verify with tokenizer.eos_token_id at runtime

# dyn_A_g05 共用参数
_DYN_A_G05_BASE = dict(
    mode='dynamic',
    n_hi=8, n_lo=3, mu_lo=4, mu_hi=16,
    gamma_n=0.5, gamma_mu=0.5, gamma_floor=1.5,
    remask_ratio_hi=0.35, floor_lo=0,
    berm_scope='trail_block',
)

CONFIGS = [
    {
        'label': 'dyn_A_g05',
        **_DYN_A_G05_BASE,
        'eos_penalty': 0.0, 'eos_token_id': None,
    },
    {
        'label': 'dyn_A_g05_EOF',
        **_DYN_A_G05_BASE,
        'eos_penalty': EOS_PENALTY, 'eos_token_id': EOS_TOKEN_ID,
    },
]

GEN_LENGTH = 256
STEPS = 256
BLOCK_LENGTH = 32
SABER_MTR = 0.8

# Runtime check: override EOS_TOKEN_ID from actual tokenizer if available
_actual_eos = getattr(tokenizer, 'eos_token_id', None)
if _actual_eos is not None and _actual_eos != EOS_TOKEN_ID:
    print(f'⚠ tokenizer.eos_token_id={_actual_eos}, overriding EOS_TOKEN_ID={EOS_TOKEN_ID}')
    EOS_TOKEN_ID = _actual_eos
    for c in CONFIGS:
        if c.get('eos_token_id') is not None:
            c['eos_token_id'] = EOS_TOKEN_ID
else:
    print(f'EOS_TOKEN_ID={EOS_TOKEN_ID} (confirmed from tokenizer)')

print(f'EOS_PENALTY={EOS_PENALTY} (only applied to _EOF config)')
print(f'{len(CONFIGS)} configs x {LIMIT} samples = {len(CONFIGS) * LIMIT} total runs')

## 5. 逐配置跑 50 samples + 生成 HTML

In [ ]:
import time

for cfg in CONFIGS:
    label = cfg['label']
    mode = cfg.get('mode', 'static')

    print(f'\n{"="*60}')
    print(f'Config: {label} (mode={mode})')
    print(f'{"="*60}')

    all_samples_json = []
    t0 = time.time()

    for i in range(LIMIT):
        prompt = prompts[i]
        sample_t0 = time.time()

        _eos_p = cfg.get('eos_penalty', 0.0)
        _eos_id = cfg.get('eos_token_id', None)

        if mode == 'dynamic':
            x, nfe, history = generate_with_collection_dynamic(
                model, prompt,
                steps=STEPS, gen_length=GEN_LENGTH, block_length=BLOCK_LENGTH,
                temperature=0.0, mask_id=MASK_ID,
                mid_trigger_ratio=SABER_MTR, global_aadu=True,
                n_hi=cfg['n_hi'], n_lo=cfg['n_lo'],
                mu_lo=cfg['mu_lo'], mu_hi=cfg['mu_hi'],
                gamma_n=cfg['gamma_n'], gamma_mu=cfg['gamma_mu'],
                gamma_floor=cfg['gamma_floor'],
                remask_ratio_hi=cfg['remask_ratio_hi'],
                floor_lo=cfg['floor_lo'],
                berm_scope=cfg.get('berm_scope', 'trail_block'),
                eos_penalty=_eos_p, eos_token_id=_eos_id,
                verbose=(i == 0), sample_idx=i,
            )
        else:
            x, nfe, history = generate_with_collection_saber(
                model, prompt,
                steps=STEPS, gen_length=GEN_LENGTH, block_length=BLOCK_LENGTH,
                temperature=0.0, mask_id=MASK_ID,
                mid_trigger_ratio=SABER_MTR,
                saber_n=cfg['saber_n'], saber_mu=cfg['saber_mu'],
                global_aadu=True, berm_scope=cfg.get('berm_scope', 'window'),
                eos_penalty=_eos_p, eos_token_id=_eos_id,
                verbose=(i == 0), sample_idx=i,
            )

        gen_text = tokenizer.decode(x[0, prompt.shape[1]:], skip_special_tokens=True)
        for stop in ['Question:', '\n\nQuestion']:
            if stop in gen_text:
                gen_text = gen_text.split(stop)[0]
        gen_ans = extract_answer(gen_text)
        ref_ans = extract_answer(ref_answers[i])
        correct = gen_ans is not None and ref_ans is not None and gen_ans == ref_ans

        tag = chr(0x2713) if correct else chr(0x2717) + ' (ref: ' + str(ref_ans) + ')'
        sample_json = process_sample_saber(
            history, tokenizer,
            gen_length=GEN_LENGTH, block_length=BLOCK_LENGTH,
            question=questions[i],
            gen_answer=f'{gen_ans} {tag}',
            ref_answer=ref_ans,
        )
        all_samples_json.append(sample_json)

        elapsed = time.time() - t0
        mark = chr(0x2713) if correct else chr(0x2717)
        print(f'  [{i+1}/{LIMIT}] NFE={nfe:3d} ans={gen_ans} {mark} ({elapsed:.1f}s)')

    correct_count = sum(1 for s in all_samples_json if chr(0x2713) in s.get('gen_answer', ''))
    total_time = time.time() - t0
    print(f'  Done! {correct_count}/{LIMIT} correct. Time: {total_time:.1f}s')

    html = generate_multi_html(
        all_samples_json,
        title=f'GSM8K Saber Denoising — {label} — {LIMIT} samples',
    )
    out_file = os.path.join(NOTEBOOK_DIR, f'viz_gsm8k_saber_{label}.html')
    with open(out_file, 'w', encoding='utf-8') as f:
        f.write(html)
    size_mb = os.path.getsize(out_file) / 1024 / 1024
    print(f'  HTML saved: {out_file} ({size_mb:.1f} MB)')

print(f'\nAll configs done!')